# Level 5 — Data Mining Challenge: *The 1,000-Pick*

**규칙**: Set B (이미지 + 라벨 공개) 에서 **최대 1,000장** 을 선택하여 학습 셋에 추가하고, best 모델을 다시 학습하세요.

> **Set B 의 라벨이 공개되어 있다는 점에 주의**하세요. 본 Level 의 평가 본질은 "*주어진 풀에서 어떤 1,000장이 가장 가치 있는가*" — 즉, 라벨을 알고 있다고 가정한 상태에서의 효율적인 부분집합 선택입니다.

**본 PA에서 가장 큰 비중 (25%)** 을 차지하는 Level 입니다. 어떤 *알고리즘* 으로 1,000장을 골랐는지 — 그 *근거* — 가 변별력의 본진입니다. Curation Report 로 정리합니다.

채점 메트릭:
$$\text{DI} = \frac{\text{Avg-MF1}(\text{본인 picks}) - \text{Avg-MF1}(\text{random picks})}{\text{Avg-MF1}(\text{random picks})}$$

## 검토해 볼 만한 전략

| 전략 | 핵심 아이디어 | Set B 라벨 활용 |
|---|---|---|
| 클래스 균형 (Class Balancing) | Set A 에서 부족한 속성 클래스 (foggy / dawn-dusk 등) 를 채워 넣음 | ✅ 라벨로 직접 필터링 |
| Hard Example Mining | base 모델의 confidence 가 낮은 / 예측이 라벨과 다른 이미지를 우선 선택 | ✅ 모델 예측 vs 정답 비교 |
| 다양성 (Core-Set) | Set B 의 feature space 를 가장 잘 커버하는 부분집합 선택 (k-center / clustering) | 라벨 무관 |
| 결합 커버리지 | 속성 *조합* 의 균형을 맞춤 — 예: (snowy & night), (rainy & residential) | ✅ 라벨로 조합 카운트 |
| Loss 기반 | Set B 이미지에 대한 학습 직전 loss 가 큰 샘플 우선 | ✅ 라벨 필요 |

위 전략들을 결합/응용/대체할 수 있습니다. **Curation Report 에 본인의 의사결정 근거를 명확히 기술** 하세요.

**산출물**: `level5_picks.json` — 선택한 image_id 리스트 (이미지별 메타데이터 포함 가능).

In [ ]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/IRCVLab/2026-HYU-AUE8088-PA2.git

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.utils.submission import write_submission
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES, NUM_CLASSES
from src.models.resnet import resnet18

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
STRATEGY_NAME = "top1k-uncertainty"   # 본인 전략명 (Run 이름에 들어감)
WANDB_TAGS    = ["level5", STRATEGY_NAME]

In [ ]:
# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨
DATA_ROOT = "../data/set_a"

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------


In [ ]:
# 1단계 — best base 모델로 Set B 의 모든 이미지를 score.
model = resnet18().to(device)
model.load_state_dict(torch.load("../checkpoints/level3_best.pth", map_location=device)["state_dict"])

set_b = BDDAttrDataset("../data/set_b", split="mining", transform=eval_transform())
loader_b = DataLoader(set_b, batch_size=128, shuffle=False, num_workers=2)

preds_b, probs_b, _, ids_b = collect_predictions(model, loader_b, device)

# 이미지별 불확실성 (uncertainty) 계산: 1 - max-softmax 를 3개 head 평균.
max_probs = np.stack([probs_b[a].max(axis=-1) for a in ATTRIBUTES], axis=1)
uncertainty = 1.0 - max_probs.mean(axis=1)
print(f"unc shape: {uncertainty.shape}, mean={uncertainty.mean():.3f}")

In [ ]:
# 2단계 — 본인의 선별 알고리즘을 설계하세요.
#
# Set B 의 정답 라벨은 set_b 의 BDDAttrDataset (split="mining") 에 이미 채워져 있습니다.
# preds_b (모델 예측) 와 set_b 의 sample.weather/scene/timeofday (정답) 를 함께 활용 가능.
#
# 아래는 placeholder (top-1000 most-uncertain). 본인 구현으로 교체하세요.
# Tip: 단일 신호보다 여러 신호를 결합하는 편이 보통 더 좋은 성능을 냅니다. 예시:
#   score_i = lam * uncertainty_i  +  (1 - lam) * is_rare_class_i
import json

K = 1000
order = np.argsort(-uncertainty)[:K]

picks = []
for i in order:
    s = set_b.samples[i]   # 정답 라벨이 채워져 있는 Sample
    picks.append({
        "image_id": s.image_id,
        "weather":   int(s.weather),    # Set B 의 정답 라벨 사용
        "scene":     int(s.scene),
        "timeofday": int(s.timeofday),
        "reason":    "placeholder: top-1k uncertainty",
    })

with open("../level5_picks.json", "w") as f:
    json.dump({
        "strategy": "이 부분을 교체하세요 — 본인의 알고리즘을 2~3 문장으로 기술",
        "num_picks": len(picks),
        "picks": picks,
    }, f, indent=2)
print(f"saved {len(picks)} picks")

In [ ]:
# 3단계 — Set A + 본인이 고른 picks 로 재학습. 학습 메트릭은 wandb 로 자동 로깅.
extra = [(p["image_id"], p["weather"], p["scene"], p["timeofday"]) for p in picks]
train_aug = BDDAttrDataset("../data/set_a", "train", transform=train_transform(), extra_picks=extra)
val_ds    = BDDAttrDataset("../data/set_a", "val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
loader_tr  = DataLoader(train_aug, batch_size=64, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
loader_val = DataLoader(val_ds,    batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

set_seed(SEED, deterministic=True)
model2 = resnet18().to(device)
optim  = torch.optim.AdamW(model2.parameters(), lr=3e-4, weight_decay=5e-4)
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=25)
losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level5-{STRATEGY_NAME}",
    config={
        "backbone": "resnet18", "strategy": STRATEGY_NAME,
        "num_picks": len(picks), "epochs": 25, "batch": 64, "lr": 3e-4, "seed": SEED,
    },
    tags=WANDB_TAGS,
)
trainer = MultiTaskTrainer(model2, optim, sched, losses, device, TrainConfig(epochs=25), logger=logger)
history = trainer.fit(loader_tr, loader_val)

# 학습 종료 후 — 속성별 confusion matrix 와 picks 분포를 wandb 에 업로드
val_pred, _, val_tgt, _ = collect_predictions(model2, loader_val, device)
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"final/cm_{a}", confusion_matrices(val_pred, val_tgt)[a], CLASS_NAMES[a])

# 본인이 고른 1,000장의 (예측) 분포 — Curation 의도가 의도대로 반영됐는지 검증
from collections import Counter
for a in ATTRIBUTES:
    cnt = Counter(p[a] for p in picks)
    rows = [[CLASS_NAMES[a][k], cnt.get(k, 0)] for k in range(NUM_CLASSES[a])]
    logger.log_table(f"picks/distribution_{a}", ["class", "count"], rows)

torch.save({"state_dict": model2.state_dict(), "history": history},
           "../checkpoints/level5_final.pth")
logger.finish()

In [ ]:
# 4단계 — Kaggle 제출용 CSV 생성.
test_ds = BDDAttrDataset("../data/set_a", "test", transform=eval_transform())
loader_te = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)

preds_te, _, _, ids_te = collect_predictions(model2, loader_te, device)
write_submission("../submission/level5_submission.csv", ids_te, preds_te)
print("submission/level5_submission.csv 생성 완료 — Kaggle 페이지에 직접 업로드 하세요.")

## Curation Report — 필수

Final PPT 에 다음을 포함하세요.
- **선별 알고리즘** (의사코드 또는 1페이지 다이어그램).
- 본인 picks 1,000장의 **분포** — (예측된) weather × scene × timeofday — 를 heatmap 또는 stacked bar 로 시각화.
- **Random-1000 baseline** 결과와 본인의 **DI score** 비교.
- **Ablation**: 250 / 500 / 1000 장을 골랐을 때의 변화 — 추가 데이터의 한계 효용이 보이는지 확인.

여러 전략을 시험했다면 wandb 의 같은 프로젝트에 `STRATEGY_NAME` 만 바꿔서 별도 Run 으로 누적하세요. 학습 곡선·분포·DI score 비교가 한 페이지에 모입니다.